# Exploración de datos Sentinel-2

**Proyecto:** Sistema Inteligente de Recomendación de Riego para el Cultivo de Patata
**Trabajo de Fin de Grado · Sprint 2 · Issues #2, #3 y #4**

## Contenido

- **Parte 1 · Estructura de los CSV (issue #2).** Análisis de los CSV de las
  cuatro parcelas, generados por el notebook
  `03_extraccion_sentinel2_parcelas_reales` para una extracción provisional del
  2026-04-01 (incluido) al 2026-10-01 (excluido). Comprueba columnas, tipos,
  valores ausentes, fechas, calidad y cobertura, y qué variables se pueden
  utilizar después.
- **Parte 2 · Ejemplo previo con maíz de 2018.** Exploración inicial con un CSV
  de **maíz de 2018**. Es un ejemplo metodológico: no contiene datos de las
  parcelas reales ni de patata.
- **Parte 3 · Evolución temporal de los índices (issue #3).** Representa NDVI,
  NDRE, NDMI y EVI de las cuatro parcelas a partir de los CSV de la Parte 1.
  Cada figura tiene un panel por parcela, distingue las observaciones
  seleccionadas de las no seleccionadas y deja visibles los huecos de
  observación. Incluye la revisión de los gráficos de la Parte 2, la tabla de
  valores por índice y parcela, y la comprobación del caso de Los Ceres.

## Orden de ejecución

- La Parte 2 es independiente y
requiere el CSV de 2018 en Drive; si no está disponible, puede omitirse sin
afectar a las otras dos.
- La Parte 3 utiliza datos y funciones definidos en la Parte 1 (`dfs_calidad`,
`esperado`, `columna_que_contiene`, `PARCELAS_ESPERADAS`, `PERIODO` y `BASE`),
por lo que debe ejecutarse después de ella.

## Límites que se mantienen en todo el notebook

- El periodo de extracción es provisional y no se confirma el cultivo presente
  en cada parcela durante ese periodo.
- Los índices Sentinel-2 describen la vegetación; no miden la humedad del suelo.
- Las Partes 1 y 3 no interpolan, no rellenan huecos y no emiten
  recomendaciones de riego.
- Las semanas sin observaciones seleccionadas conservan valores ausentes.

# Parte 1 - CSV de las cuatro parcelas (2026, periodo provisional)

## 1.1 Configuración y acceso a Drive

Si el notebook se ejecuta fuera de Colab, se omite el montaje de Drive y hay que
ajustar `BASE` a una ruta local.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fuera de Colab: no se monta Drive. Ajusta BASE a una ruta local.")

BASE = Path("/content/drive/MyDrive/TFG_PATATAS_TRABAJO")
PERIODO = ("2026-04-01", "2026-10-01")
PARCELAS_ESPERADAS = ["pancostales", "puri", "la_culebra", "los_ceres"]
RESULTADOS = BASE / "resultados" / "analisis_estructura_csv"

print("Carpeta base:", BASE, "| existe:", BASE.exists())

In [ ]:
import datetime as dt

carpeta_lotes = BASE / "resultados" / "sentinel2_lotes"
print("¿Existe la carpeta de lotes?", carpeta_lotes.exists())

for ruta in sorted(carpeta_lotes.rglob("*.csv")):
    info = ruta.stat()
    print(f"{ruta.relative_to(carpeta_lotes).as_posix():<110} "
          f"{info.st_size/1024:8.1f} KB  {dt.datetime.fromtimestamp(info.st_mtime):%Y-%m-%d %H:%M}")

## 1.2 Localizar los CSV de cada parcela

Se buscan tres tipos de archivo generados por el notebook 03 para el periodo
indicado: el CSV de índices (uno por imagen con píxeles válidos), el CSV de
revisión de calidad y el resumen semanal. Si falta alguna parcela o hay archivos
duplicados, el notebook se detiene para no mezclar ejecuciones distintas.

In [ ]:
sufijo_periodo = f"{PERIODO[0]}_{PERIODO[1]}"
carpeta_lotes = BASE / "resultados" / "sentinel2_lotes"

def buscar(patron, excluir=()):
    encontrados = {p: [] for p in PARCELAS_ESPERADAS}
    for ruta in sorted(carpeta_lotes.rglob(patron)):
        if sufijo_periodo not in ruta.name or any(t in ruta.name for t in excluir):
            continue
        for p in PARCELAS_ESPERADAS:
            if ruta.name.startswith(f"{p}_sentinel2_"):
                encontrados[p].append(ruta)
    return encontrados

hallados = {
    "indices": buscar("*_sentinel2_indices_*.csv", excluir=("calidad",)),
    "calidad": buscar("*_sentinel2_indices_calidad_*.csv"),
    "semanal": buscar("*_sentinel2_semanal_*.csv"),
}

for tipo, por_parcela in hallados.items():
    faltan = [p for p, rutas in por_parcela.items() if not rutas]
    repetidos = {p: [r.name for r in rutas] for p, rutas in por_parcela.items() if len(rutas) > 1}
    if faltan:
        raise FileNotFoundError(f"Faltan CSV de tipo '{tipo}' para: {faltan}. Revisa BASE y los nombres.")
    if repetidos:
        raise ValueError(f"Hay varios archivos de tipo '{tipo}' para una parcela: {repetidos}")

csv_indices = {p: hallados["indices"][p][0] for p in PARCELAS_ESPERADAS}
csv_calidad = {p: hallados["calidad"][p][0] for p in PARCELAS_ESPERADAS}
csv_semanal = {p: hallados["semanal"][p][0] for p in PARCELAS_ESPERADAS}

tabla_archivos = pd.DataFrame({
    "parcela": PARCELAS_ESPERADAS,
    "csv_indices": [csv_indices[p].relative_to(BASE).as_posix() for p in PARCELAS_ESPERADAS],
})
display(tabla_archivos)

## 1.3 Carga y resumen por parcela

Se distingue entre **filas** (una por imagen con píxeles válidos), **imágenes
distintas** y **fechas distintas**. Una parcela situada en dos teselas puede
tener varias imágenes en la misma fecha.

In [ ]:
def columna_imagen(df):
    return next((c for c in df.columns if c.lower().replace("_", "") == "imageid"), None)

dfs = {
    p: pd.read_csv(ruta, parse_dates=["date"], encoding="utf-8-sig")
    for p, ruta in csv_indices.items()
}

filas = []
for p, df_p in dfs.items():
    col_img = columna_imagen(df_p)
    por_fecha = df_p["date"].value_counts()
    filas.append({
        "parcela": p,
        "filas": len(df_p),
        "columnas": df_p.shape[1],
        "imagenes_distintas": df_p[col_img].nunique() if col_img else None,
        "fechas_distintas": df_p["date"].nunique(),
        "fechas_con_varias_filas": int((por_fecha > 1).sum()),
        "primera_fecha": df_p["date"].min().strftime("%Y-%m-%d"),
        "ultima_fecha": df_p["date"].max().strftime("%Y-%m-%d"),
        "celdas_nulas": int(df_p.isna().sum().sum()),
    })

resumen = pd.DataFrame(filas)
display(resumen)

## 1.4 Columnas: comparación entre parcelas

Las cuatro parcelas deben compartir exactamente las mismas columnas y en el mismo
orden. Cualquier diferencia se muestra en las columnas `faltan` y `sobran`.

In [ ]:
parcela_ref = PARCELAS_ESPERADAS[0]
columnas_ref = list(dfs[parcela_ref].columns)

comparacion_columnas = pd.DataFrame([
    {
        "parcela": p,
        "mismas_columnas_y_orden": list(df_p.columns) == columnas_ref,
        "faltan": sorted(set(columnas_ref) - set(df_p.columns)),
        "sobran": sorted(set(df_p.columns) - set(columnas_ref)),
    }
    for p, df_p in dfs.items()
])
display(comparacion_columnas)

print(f"Columnas de la parcela de referencia ({parcela_ref}): {len(columnas_ref)}")
for numero, columna in enumerate(columnas_ref, start=1):
    print(f"{numero:02d}. {columna}")

## 1.5 Agrupación de columnas y tipos de datos

Las estadísticas siguen el patrón `ÍNDICE_estadístico` (por ejemplo `NDVI_mean`).
Revisa la columna `grupo`: lo que aparezca como `otras` no está clasificado y
conviene explicarlo en el documento de análisis.

In [ ]:
SUFIJOS = ["mean", "stdDev", "std", "min", "max", "median", "p25", "p75", "count"]
indices = sorted({c[: -len("_mean")] for c in columnas_ref if c.endswith("_mean")})

def grupo_columna(columna):
    for indice in indices:
        if columna.startswith(indice + "_") and columna[len(indice) + 1:] in SUFIJOS:
            return f"estadistica_{columna[len(indice) + 1:]}"
    k = columna.lower()
    if k == "date" or k.endswith("id") or any(t in k for t in ("nombre", "municipio", "variedad")):
        return "identificacion"
    if any(t in k for t in ("cloud", "pixel", "conservado", "cobertura")):
        return "calidad"
    if any(t in k for t in ("periodo", "cultivo", "fecha_", "escala", "umbral")):
        return "contexto_extraccion"
    return "otras"

estructura = pd.DataFrame({
    "columna": columnas_ref,
    "grupo": [grupo_columna(c) for c in columnas_ref],
    "tipo": dfs[parcela_ref].dtypes.astype(str).values,
    "nulos_en_referencia": dfs[parcela_ref].isna().sum().values,
})

print("Índices detectados:", indices)
display(estructura.groupby("grupo").size().rename("n_columnas").to_frame())
display(estructura)

tipos_por_parcela = pd.DataFrame({p: df_p.dtypes.astype(str) for p, df_p in dfs.items()})
diferencias_tipos = tipos_por_parcela[tipos_por_parcela.nunique(axis=1) > 1]
if diferencias_tipos.empty:
    print("Los tipos de datos coinciden en las cuatro parcelas.")
else:
    display(diferencias_tipos)

## 1.6 Valores ausentes

Se cuentan los valores nulos por columna y parcela. En el CSV de índices cada fila
procede de una imagen con píxeles válidos, de modo que se esperan pocos nulos. Los
huecos temporales no aparecen aquí: se estudian en el apartado siguiente.

In [ ]:
nulos_por_columna = pd.DataFrame({p: df_p.isna().sum() for p, df_p in dfs.items()})
nulos_con_valores = nulos_por_columna[nulos_por_columna.sum(axis=1) > 0]

if nulos_con_valores.empty:
    print("No hay valores nulos en ninguna columna de las cuatro parcelas.")
else:
    display(nulos_con_valores)

## 1.7 Fechas, huecos e imágenes repetidas

Se calculan los intervalos entre fechas distintas y se listan los huecos de 15 días
o más. También se comprueba qué fechas tienen más de una fila.

In [ ]:
UMBRAL_HUECO_DIAS = 15

filas_fechas, huecos = [], []
for p, df_p in dfs.items():
    fechas = df_p["date"].drop_duplicates().sort_values().reset_index(drop=True)
    intervalos = fechas.diff().dt.days.dropna()
    filas_fechas.append({
        "parcela": p,
        "fechas_distintas": len(fechas),
        "intervalo_minimo_dias": intervalos.min(),
        "intervalo_mediano_dias": intervalos.median(),
        "intervalo_medio_dias": round(intervalos.mean(), 2),
        "intervalo_maximo_dias": intervalos.max(),
    })
    for posicion in intervalos[intervalos >= UMBRAL_HUECO_DIAS].index:
        huecos.append({
            "parcela": p,
            "desde": fechas[posicion - 1].strftime("%Y-%m-%d"),
            "hasta": fechas[posicion].strftime("%Y-%m-%d"),
            "dias": int(intervalos[posicion]),
        })

resumen_fechas = pd.DataFrame(filas_fechas)
huecos_largos = pd.DataFrame(huecos)
display(resumen_fechas)
print(f"Huecos de {UMBRAL_HUECO_DIAS} días o más entre fechas con imagen válida:")
display(huecos_largos if not huecos_largos.empty else "Ninguno")

repetidas = []
for p, df_p in dfs.items():
    col_img = columna_imagen(df_p)
    conteo = df_p.groupby("date").size()
    for fecha, n in conteo[conteo > 1].items():
        repetidas.append({
            "parcela": p,
            "fecha": fecha.strftime("%Y-%m-%d"),
            "filas": int(n),
            "imagenes": sorted(df_p.loc[df_p["date"] == fecha, col_img].astype(str)) if col_img else None,
        })
fechas_repetidas = pd.DataFrame(repetidas)
print("Fechas con más de una fila en el CSV de índices:")
display(fechas_repetidas.head(10) if not fechas_repetidas.empty else "Ninguna")
if not fechas_repetidas.empty:
    print("Total de fechas repetidas por parcela:")
    display(fechas_repetidas.groupby("parcela").size().rename("fechas_repetidas").to_frame())

## 1.8 Indicadores de calidad y cobertura

El CSV de índices conserva todas las imágenes con algún píxel válido, no solo las
buenas. Para decidir cuáles se usan, el notebook 03 aplica una **cobertura mínima
del 70 % por índice** y exige medias completas (`seleccion_tecnica_provisional`).
Ese criterio es distinto del filtro orientativo de la Parte 2 (50 % del máximo de
`NDVI_count`) y no deben mezclarse.

In [ ]:
cols_calidad = ["cloud_percentage", "pixeles_antes", "pixeles_despues", "porcentaje_conservado"]
cols_count = [c for c in columnas_ref if c.endswith("_count")]

descripcion_calidad = pd.concat(
    {
        p: df_p[[c for c in cols_calidad if c in df_p.columns]].describe().T
        for p, df_p in dfs.items()
    },
    names=["parcela", "variable"],
)
display(descripcion_calidad.round(2))
print("Columnas de recuento de píxeles por índice:", cols_count)

def columna_que_contiene(df, *claves):
    return next(
        (c for c in df.columns if all(k in c.lower().replace("_", "") for k in claves)),
        None,
    )

dfs_calidad = {p: pd.read_csv(r, parse_dates=["date"], encoding="utf-8-sig") for p, r in csv_calidad.items()}

filas_sel = []
for p, dfq in dfs_calidad.items():
    col_sel = columna_que_contiene(dfq, "seleccion")
    col_cob = columna_que_contiene(dfq, "coberturaminima")
    seleccionadas = int(dfq[col_sel].astype(bool).sum()) if col_sel else None
    filas_sel.append({
        "parcela": p,
        "observaciones_revisadas": len(dfq),
        "seleccionadas": seleccionadas,
        "no_seleccionadas": len(dfq) - seleccionadas if seleccionadas is not None else None,
        "cobertura_minima_p50": round(dfq[col_cob].median(), 2) if col_cob else None,
    })
seleccion_provisional = pd.DataFrame(filas_sel)
display(seleccion_provisional)

## 1.9 Resumen semanal sin interpolación

El CSV semanal agrupa de lunes (incluido) a lunes siguiente (excluido) y calcula la
media de las medias de las observaciones seleccionadas. Las semanas sin
observaciones conservan valores ausentes.

In [ ]:
filas_sem = []
for p, ruta in csv_semanal.items():
    dsem = pd.read_csv(ruta, encoding="utf-8-sig")
    col_obs = columna_que_contiene(dsem, "tieneobservaciones")
    con_datos = int(dsem[col_obs].astype(bool).sum()) if col_obs else None
    filas_sem.append({
        "parcela": p,
        "semanas_totales": len(dsem),
        "semanas_con_observaciones": con_datos,
        "semanas_sin_observaciones": len(dsem) - con_datos if con_datos is not None else None,
    })
resumen_semanal = pd.DataFrame(filas_sem)
display(resumen_semanal)

## 1.10 Contraste con la ejecución del notebook 03

Valores obtenidos en la ejecución del notebook 03 con el mismo periodo y los mismos
parámetros (umbral de nube 20, cobertura mínima 70 %, escala 20 m). Si alguno no
coincide, hay que registrar la discrepancia y su causa antes de documentar.

In [ ]:
esperado = pd.DataFrame([
    {"parcela": "pancostales", "filas": 39, "fechas_distintas": 39, "seleccionadas": 25, "semanas_sin_observaciones": 10},
    {"parcela": "puri",        "filas": 68, "fechas_distintas": 35, "seleccionadas": 29, "semanas_sin_observaciones": 7},
    {"parcela": "la_culebra", "filas": 39, "fechas_distintas": 39, "seleccionadas": 27, "semanas_sin_observaciones": 9},
    {"parcela": "los_ceres",  "filas": 35, "fechas_distintas": 35, "seleccionadas": 26, "semanas_sin_observaciones": 9},
]).set_index("parcela")

obtenido = (
    resumen.set_index("parcela")[["filas", "fechas_distintas"]]
    .join(seleccion_provisional.set_index("parcela")[["seleccionadas"]])
    .join(resumen_semanal.set_index("parcela")[["semanas_sin_observaciones"]])
)

contraste = esperado.join(obtenido, lsuffix="_esperado", rsuffix="_obtenido")
for columna in esperado.columns:
    contraste[f"{columna}_coincide"] = contraste[f"{columna}_esperado"] == contraste[f"{columna}_obtenido"]
display(contraste)
print("Todas las comprobaciones coinciden:", bool(contraste.filter(like="_coincide").all().all()))

## 1.11 Índices disponibles y variables utilizables

Se comprueba qué índices solicitados en el plan existen realmente en la extracción
de 2026. NDRE se incorporó al extractor el 22/10/2026 (issue #4), por lo que debe aparecer como disponible.

In [ ]:
INDICES_PLAN = ["NDVI", "NDRE", "NDMI", "MSI", "EVI", "SAVI"]
disponibilidad = pd.DataFrame({
    "indice": INDICES_PLAN,
    "columna_media": [f"{i}_mean" for i in INDICES_PLAN],
    "disponible_en_2026": [f"{i}_mean" in columnas_ref for i in INDICES_PLAN],
})
display(disponibilidad)

ausentes = disponibilidad.loc[~disponibilidad["disponible_en_2026"], "indice"].tolist()
print("Índices del plan que NO están en la extracción de 2026:", ausentes or "ninguno")

| Elemento | ¿Lo aporta esta extracción? | Uso posterior |
|---|---|---|
| NDVI, NDMI, MSI, EVI, SAVI por imagen | Sí | Series por parcela; NDVI puede alimentar la estimación de Kc |
| Calidad y cobertura por imagen | Sí | Selección de observaciones |
| Resumen semanal sin interpolar | Sí | Entrada temporal para el simulador |
| Humedad del suelo, meteorología, riegos | No | Fuentes externas pendientes |
| Cultivo y fechas agronómicas confirmadas | No | Periodo provisional |

## 1.12 Guardar las tablas como evidencia

Se guardan las tablas de esta parte en Drive. Después, conserva las salidas del
notebook al subirlo a GitHub, pero **borra cualquier código de verificación o
token** que haya podido quedar en otras celdas.

In [ ]:
RESULTADOS.mkdir(parents=True, exist_ok=True)

tablas = {
    "resumen_parcelas.csv": resumen,
    "comparacion_columnas.csv": comparacion_columnas,
    "estructura_columnas.csv": estructura,
    "resumen_fechas.csv": resumen_fechas,
    "huecos_largos.csv": huecos_largos,
    "fechas_repetidas.csv": fechas_repetidas,
    "seleccion_provisional.csv": seleccion_provisional,
    "resumen_semanal.csv": resumen_semanal,
    "contraste_notebook_03.csv": contraste.reset_index(),
    "disponibilidad_indices.csv": disponibilidad,
}

for nombre, tabla in tablas.items():
    tabla.to_csv(RESULTADOS / nombre, index=False, encoding="utf-8-sig")

print("Tablas guardadas en:", RESULTADOS)
for nombre in tablas:
    print(" -", nombre)

# Parte 2 - Exploración previa con un CSV de maíz de 2018 (ejemplo metodológico)

Esta parte conserva la exploración inicial realizada con un único CSV de maíz de
2018. **No son datos de las parcelas reales ni de patata.** Sirve como ejemplo del
análisis y del tratamiento de observaciones repetidas por fecha.

Requiere que la carpeta `Sentinel2_Indices` esté en Drive. Si no existe, se puede
omitir esta parte.

## 2.1 Fuente de datos

Los archivos fueron generados con Google Earth Engine y almacenados en
`/content/drive/MyDrive/Sentinel2_Indices/`. El archivo principal es
`maiz_sentinel2_indices_abril_septiembre_2018.csv`.

## 2.2 Carga del conjunto de datos

Se convierte la columna `date` a formato de fecha para poder ordenar y analizar
temporalmente las observaciones.

In [ ]:
drive_path = "/content/drive/MyDrive/Sentinel2_Indices"
csv_path = os.path.join(drive_path, "maiz_sentinel2_indices_abril_septiembre_2018.csv")

print("¿Existe la carpeta?", os.path.exists(drive_path))
print("¿Existe el CSV?", os.path.exists(csv_path))

if not os.path.exists(csv_path):
    raise FileNotFoundError("No se encuentra el CSV de 2018. Omite la Parte 2 o revisa la ruta.")

print("\nArchivos encontrados en la carpeta:")
for file in sorted(os.listdir(drive_path)):
    print(file)

In [ ]:
df = pd.read_csv(csv_path, parse_dates=["date"])

print("Archivo cargado correctamente.")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df = df.sort_values("date").reset_index(drop=True)
print("Primera fecha:", df["date"].min())
print("Última fecha:", df["date"].max())
print("Número de fechas distintas:", df["date"].nunique())

df.head()

## 2.3 Estructura del conjunto de datos

Se comprueba el número de observaciones, las variables disponibles y los tipos de
datos del archivo.

In [ ]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])
print("Dimensiones del DataFrame:", df.shape)

print("\nColumnas disponibles:\n")
for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

In [ ]:
column_groups = {
    "Identificacion": [c for c in df.columns if c in ["date", "image_id", "cloud_percentage"]],
    "Medias": [c for c in df.columns if c.endswith("_mean")],
    "Desviaciones": [c for c in df.columns if c.endswith("_std")],
    "Minimos": [c for c in df.columns if c.endswith("_min")],
    "Maximos": [c for c in df.columns if c.endswith("_max")],
    "Medianas": [c for c in df.columns if c.endswith("_median")],
    "Percentiles_25": [c for c in df.columns if c.endswith("_p25")],
    "Percentiles_75": [c for c in df.columns if c.endswith("_p75")],
    "Conteos": [c for c in df.columns if c.endswith("_count")],
}

for group_name, columns in column_groups.items():
    print(f"\n{group_name} ({len(columns)} columnas):")
    print(columns)

## 2.4 Tipos de datos

Se comprueba que las fechas se han convertido correctamente y que las variables
numéricas se pueden utilizar en los análisis posteriores.

In [ ]:
print("Tipo de la columna date:", df["date"].dtype)

for column in ["NDVI_mean", "NDVI_std", "NDVI_count", "cloud_percentage"]:
    if column in df.columns:
        print(column, "->", df[column].dtype)

types_table = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values})
types_table

## 2.5 Análisis temporal

Se estudia el intervalo de fechas disponible y el número de observaciones por
fecha para comprobar la frecuencia real de los datos Sentinel-2.

In [ ]:
print("Primera fecha:", df["date"].min())
print("Última fecha:", df["date"].max())
print("Número total de filas:", len(df))
print("Número de fechas distintas:", df["date"].nunique())

date_counts = df["date"].value_counts().sort_index()
print("\nFechas con más de una observación:", int((date_counts > 1).sum()))
display(date_counts[date_counts > 1].head(10))

unique_dates = df["date"].drop_duplicates().sort_values()
date_gaps = unique_dates.diff().dt.days.dropna()

print("\nIntervalos entre observaciones, en días:")
print(date_gaps.to_list())
print("Intervalo mínimo:", date_gaps.min(), "días")
print("Intervalo máximo:", date_gaps.max(), "días")
print("Intervalo medio:", round(date_gaps.mean(), 2), "días")
print("Mediana del intervalo:", date_gaps.median(), "días")

## 2.6 Valores nulos

Se comprueba si existen valores nulos en las variables generadas después de aplicar
la máscara de nubes y calcular las estadísticas espaciales.

In [ ]:
null_summary = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_percentage": df.isna().mean() * 100,
}).sort_values("null_percentage", ascending=False)

display(null_summary.head(30))

for column in ["date", "NDVI_mean", "NDVI_count", "NDRE_mean", "NDMI_mean", "EVI_mean"]:
    if column in df.columns:
        print(column, "-> valores nulos:", df[column].isna().sum())

## 2.7 Estadísticas descriptivas

Se calculan estadísticas básicas de los principales índices para conocer sus rangos
y detectar posibles valores anómalos.

In [ ]:
mean_columns = [c for c in df.columns if c.endswith("_mean")]
print("Columnas de medias:")
print(mean_columns)

descriptive_statistics = df[mean_columns].describe().T
display(descriptive_statistics)

selected_means = [c for c in ["NDVI_mean", "GNDVI_mean", "NDRE_mean", "NDMI_mean", "EVI_mean"] if c in df.columns]
display(df[selected_means].describe().T)

## 2.8 Calidad de las observaciones con NDVI_count

`NDVI_count` es el número de píxeles válidos usados en las estadísticas del NDVI
tras aplicar la máscara de nubes. Se usa como indicador aproximado de la cantidad
de información disponible en cada fecha; **no es un porcentaje de nubosidad**.

El umbral del 50 % del máximo es orientativo y solo se aplica a este ejemplo.

In [ ]:
display(df["NDVI_count"].describe())

ndvi_count_max = df["NDVI_count"].max()
quality_threshold = 0.5 * ndvi_count_max
valid_quality = df["NDVI_count"] >= quality_threshold

print("Máximo NDVI_count:", ndvi_count_max)
print("Umbral orientativo:", quality_threshold)
print("Observaciones totales:", len(df))
print("Observaciones válidas:", int(valid_quality.sum()))
print("Observaciones descartadas:", int((~valid_quality).sum()))

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(df["date"], df["NDVI_count"], color="steelblue", alpha=0.75, label="NDVI_count")
plt.axhline(quality_threshold, color="red", linestyle="--", label="Umbral del 50 %")
plt.xlabel("Fecha")
plt.ylabel("Píxeles válidos")
plt.title("Píxeles válidos por fecha (ejemplo de maíz, 2018)")
plt.legend()
plt.grid(axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2.9 Evolución temporal del NDVI

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["NDVI_mean"], marker="o", linewidth=1.8, color="forestgreen", label="NDVI medio")
plt.xlabel("Fecha")
plt.ylabel("NDVI")
plt.title("Evolución temporal del NDVI (ejemplo de maíz, 2018)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["NDVI_mean"], color="gray", linewidth=1.2, alpha=0.7)
plt.scatter(df.loc[valid_quality, "date"], df.loc[valid_quality, "NDVI_mean"],
            color="green", s=70, label="Calidad suficiente")
plt.scatter(df.loc[~valid_quality, "date"], df.loc[~valid_quality, "NDVI_mean"],
            color="red", s=70, label="Calidad baja")
plt.xlabel("Fecha")
plt.ylabel("NDVI medio")
plt.title("NDVI según la calidad de la observación (ejemplo de maíz, 2018)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2.10 Índices principales

In [ ]:
available_indices = [c for c in ["NDVI_mean", "NDRE_mean", "NDMI_mean", "EVI_mean"] if c in df.columns]

plt.figure(figsize=(13, 6))
for column in available_indices:
    plt.plot(df["date"], df[column], marker="o", linewidth=1.5, label=column)
plt.xlabel("Fecha")
plt.ylabel("Valor del índice")
plt.title("Evolución temporal de índices Sentinel-2 (ejemplo de maíz, 2018)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2.11 Guardar figura y estadísticas del ejemplo de 2018

In [ ]:
results_path = os.path.join(drive_path, "resultados_exploracion")
os.makedirs(results_path, exist_ok=True)

ndvi_figure_path = os.path.join(results_path, "evolucion_ndvi.png")

plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["NDVI_mean"], marker="o", linewidth=1.8, color="forestgreen")
plt.xlabel("Fecha")
plt.ylabel("NDVI medio")
plt.title("Evolución temporal del NDVI (ejemplo de maíz, 2018)")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(ndvi_figure_path, dpi=300, bbox_inches="tight")
plt.show()

statistics_path = os.path.join(results_path, "estadisticas_descriptivas_indices.csv")
descriptive_statistics.to_csv(statistics_path, encoding="utf-8-sig")

print("Figura guardada en:", ndvi_figure_path)
print("Estadísticas guardadas en:", statistics_path)

## 2.12 Conclusiones preliminares del ejemplo

- El CSV contiene observaciones procedentes de imágenes Sentinel-2, con dos filas
  por fecha (dos teselas).
- La serie presenta observaciones en fechas concretas, no una observación diaria.
- Los índices espectrales permiten analizar la evolución de la vegetación.
- `NDVI_count` es un indicador aproximado de calidad espacial.
- Las imágenes con pocos píxeles válidos deben estudiarse antes de incorporarlas
  al simulador.
- Sentinel-2 no proporciona directamente la humedad del suelo; hará falta
  información meteorológica y un modelo de balance hídrico.

## 2.13 Agrupación por fecha y filtro orientativo (ejemplo)

Algunas fechas tienen dos filas porque el polígono cae en dos teselas. Aquí se
agrupan por fecha calculando la **media simple** de las medias y se aplica de nuevo
el filtro orientativo sobre `NDVI_count`.

Esta agrupación **no pondera por píxeles** y **no interpola**: no se genera una serie
diaria ni valores estimados. La decisión sobre interpolación y sobre cómo resolver
varias imágenes por fecha se toma en la issue #4. La extracción de 2026 conserva una
sola imagen seleccionada por parcela y fecha y no interpola.

In [ ]:
df = df.sort_values("date").reset_index(drop=True)

date_counts = df["date"].value_counts().sort_index()
number_of_unique_dates = df["date"].nunique()
dates_with_multiple_observations = int((date_counts > 1).sum())
rows_in_multiple_observation_dates = int(date_counts[date_counts > 1].sum())

ndvicount_max = df["NDVI_count"].max()
row_threshold = 0.5 * ndvicount_max
valid_rows_count = int((df["NDVI_count"] >= row_threshold).sum())

columns_for_grouping = [c for c in df.columns if c.endswith("_mean")]
if "NDVI_count" in df.columns:
    columns_for_grouping.append("NDVI_count")

daily_observed = df.groupby("date")[columns_for_grouping].mean().sort_index()

grouped_count_max = daily_observed["NDVI_count"].max()
grouped_threshold = 0.5 * grouped_count_max
valid_grouped_quality = daily_observed["NDVI_count"] >= grouped_threshold
daily_quality = daily_observed[valid_grouped_quality].copy()

summary = pd.DataFrame({
    "Métrica": [
        "Número de filas originales",
        "Número de columnas",
        "Número de fechas distintas",
        "Primera fecha",
        "Última fecha",
        "Fechas con múltiples observaciones",
        "Filas pertenecientes a fechas múltiples",
        "Valores nulos en NDVI_mean",
        "NDVI_count máximo por fila",
        "Umbral de calidad por fila",
        "Filas válidas antes de agrupar",
        "Filas después de agrupar",
        "NDVI_count máximo agrupado",
        "Umbral de calidad agrupado",
        "Fechas válidas después de agrupar",
    ],
    "Valor": [
        len(df),
        df.shape[1],
        number_of_unique_dates,
        df["date"].min(),
        df["date"].max(),
        dates_with_multiple_observations,
        rows_in_multiple_observation_dates,
        int(df["NDVI_mean"].isna().sum()),
        ndvicount_max,
        row_threshold,
        valid_rows_count,
        len(daily_observed),
        grouped_count_max,
        grouped_threshold,
        int(valid_grouped_quality.sum()),
    ],
})
display(summary)

summary_path = os.path.join(results_path, "resumen_exploracion.csv")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("Resumen guardado en:", summary_path)

print("\nComprobaciones:")
print("Filas originales:", len(df))
print("Fechas distintas:", df["date"].nunique())
print("Filas después de agrupar:", len(daily_observed))
print("Fechas duplicadas después de agrupar:", int(daily_observed.index.duplicated().sum()))
print("Fechas conservadas después del filtro:", len(daily_quality))

## 2.14 Conclusiones de la agrupación (ejemplo)

- Se ha obtenido una única fila por fecha mediante la media simple de las medias.
- Se ha aplicado un filtro orientativo basado en `NDVI_count`.
- **No** se ha generado una serie diaria ni se ha interpolado ningún valor.
- La agrupación no pondera por número de píxeles de cada tesela; es una
  simplificación del ejemplo.
- El tratamiento definitivo de observaciones repetidas e interpolación se decide
  en la issue #4.

# Parte 3 - Evolución temporal de los índices (issue #3)

Se representan NDVI, NDRE, NDMI y EVI de las cuatro parcelas para el periodo
provisional 01/04/2026–01/10/2026. Cada figura tiene un panel por parcela,
con el mismo eje vertical. Se dibujan las observaciones seleccionadas
(línea y puntos) y, en gris, las no seleccionadas. La línea se corta
cuando entre dos observaciones seleccionadas pasan más de 14 días, para
que los huecos se vean. No se interpola. NDRE se incorporó al extractor en la issue #4 y se representa igual que el resto.

## Revisión de los gráficos existentes

Los gráficos de la Parte 2 usan un único CSV de maíz de 2018.

| Gráfico existente | Valoración | Decisión |
|---|---|---|
| Píxeles válidos por fecha | Usa el criterio de 2018 (50 % del máximo de `NDVI_count`) | No se reutiliza |
| NDVI y NDVI según calidad | Una parcela, sin periodo provisional | Se conserva como ejemplo |
| Cuatro índices en un eje | Mezcla escalas distintas | Se sustituye por una figura por índice |
| Figura guardada del NDVI | Solo maíz 2018 | Se conserva como ejemplo |


## 3.1 Configuración y criterios de representación

Se fijan los parámetros de las figuras. Se representan NDVI, NDRE, NDMI y EVI: son los índices que pide la issue y que existen en la extracción de 2026. NDRE se incorporó al extractor en la issue #4 y se representa igual que el resto.

`UMBRAL_CORTE_DIAS` indica cuántos días sin observación seleccionada se
admiten antes de cortar la línea. Es un criterio de visualización para que los
huecos se vean, no un criterio agronómico. Las figuras se guardan en la
carpeta `graficas_evolucion_temporal` de Drive.

In [ ]:
INDICES_GRAFICAS = ["NDVI", "NDRE", "NDMI", "EVI"]
NOMBRES = {"pancostales": "Pancostales", "puri": "Puri", "la_culebra": "La Culebra", "los_ceres": "Los Ceres"}
UMBRAL_CORTE_DIAS = 14

CARPETA_GRAFICAS = BASE / "resultados" / "graficas_evolucion_temporal"
CARPETA_GRAFICAS.mkdir(parents=True, exist_ok=True)

## 3.2 Preparación de las series y comprobación de cifras

Se parte de los CSV de revisión de calidad generados por el notebook 03, que
contienen todas las imágenes con algún píxel válido, las estadísticas de cada
índice y la marca de selección técnica provisional.

Cada tabla se ordena por fecha y la marca de selección se convierte a valor
lógico. Antes de graficar se comprueba que existe la media de cada índice y que
el número de observaciones seleccionadas de cada parcela coincide con el del
notebook 03: 25 en Pancostales, 29 en Puri, 27 en La Culebra y 26 en Los Ceres.
Si alguna cifra no coincide, la ejecución se detiene.


In [ ]:
series = {}
for parcela, tabla in dfs_calidad.items():
    tabla = tabla.sort_values("date").reset_index(drop=True).copy()
    col_seleccion = columna_que_contiene(tabla, "seleccion")
    tabla["seleccionada"] = tabla[col_seleccion].astype(bool)
    series[parcela] = tabla

for indice in INDICES_GRAFICAS:
    assert all(f"{indice}_mean" in t.columns for t in series.values()), f"Falta {indice}_mean"

for parcela in PARCELAS_ESPERADAS:
    n_sel = int(series[parcela]["seleccionada"].sum())
    assert n_sel == esperado.loc[parcela, "seleccionadas"], f"{parcela}: {n_sel} seleccionadas"

resumen_series = pd.DataFrame({
    "parcela": PARCELAS_ESPERADAS,
    "observaciones": [len(series[p]) for p in PARCELAS_ESPERADAS],
    "seleccionadas": [int(series[p]["seleccionada"].sum()) for p in PARCELAS_ESPERADAS],
})
display(resumen_series)

## 3.3 Figuras de evolución temporal

Se genera una figura por índice con un panel por parcela y el mismo eje
vertical en los cuatro paneles, para poder compararlos.

- Los puntos verdes unidos por una línea son las observaciones seleccionadas.
- Los puntos grises son las observaciones no seleccionadas; se muestran sin
  línea para ver qué se ha descartado.
- La línea se corta cuando entre dos observaciones seleccionadas pasan más de
  14 días. No se interpola ni se rellena ningún valor.
- El eje horizontal cubre el periodo provisional, del 01/04/2026 al 01/10/2026.

El valor representado es la media del índice sobre los píxeles válidos de la
parcela en cada imagen (`<ÍNDICE>_mean`).




In [ ]:
def serie_con_cortes(tabla_sel, columna, umbral_dias):
    fechas = tabla_sel["date"].reset_index(drop=True)
    valores = tabla_sel[columna].reset_index(drop=True)
    xs, ys = [], []
    for i in range(len(tabla_sel)):
        if i > 0 and (fechas[i] - fechas[i - 1]).days > umbral_dias:
            xs.append(fechas[i - 1] + (fechas[i] - fechas[i - 1]) / 2)
            ys.append(np.nan)
        xs.append(fechas[i])
        ys.append(valores[i])
    return xs, ys

def grafica_indice(indice):
    columna = f"{indice}_mean"
    todos = pd.concat([series[p][columna] for p in PARCELAS_ESPERADAS])
    margen = 0.05 * (todos.max() - todos.min())

    fig, ejes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    for eje, parcela in zip(ejes.ravel(), PARCELAS_ESPERADAS):
        tabla = series[parcela]
        sel = tabla[tabla["seleccionada"]]
        no_sel = tabla[~tabla["seleccionada"]]
        eje.scatter(no_sel["date"], no_sel[columna], color="lightgray", s=25,
                    label="Observación no seleccionada")
        xs, ys = serie_con_cortes(sel, columna, UMBRAL_CORTE_DIAS)
        eje.plot(xs, ys, marker="o", linewidth=1.6, color="forestgreen",
                 label="Observación seleccionada")
        eje.set_title(NOMBRES[parcela])
        eje.set_xlim(pd.Timestamp(PERIODO[0]), pd.Timestamp(PERIODO[1]))
        eje.set_ylim(todos.min() - margen, todos.max() + margen)
        eje.grid(True)
        eje.tick_params(axis="x", rotation=45)

    for eje in ejes[1]:
        eje.set_xlabel("Fecha")
    for eje in ejes[:, 0]:
        eje.set_ylabel(f"{indice} (media de la parcela)")

    handles, etiquetas = ejes[0, 0].get_legend_handles_labels()
    fig.legend(handles, etiquetas, loc="lower center", ncol=2)
    fig.suptitle(f"Evolución temporal del {indice} · periodo provisional "
                 f"{PERIODO[0]} a {PERIODO[1]}", fontsize=13)
    fig.tight_layout(rect=[0, 0.05, 1, 0.95])

    ruta = CARPETA_GRAFICAS / f"evolucion_{indice.lower()}_cuatro_parcelas.png"
    fig.savefig(ruta, dpi=200, bbox_inches="tight")
    plt.show()
    return ruta

rutas_graficas = [grafica_indice(i) for i in INDICES_GRAFICAS]

## 3.4 Huecos de observación

Se listan los intervalos de más de 14 días entre observaciones seleccionadas
consecutivas. Estos son los tramos en los que las figuras no tienen línea. La
tabla se guarda como evidencia en `huecos_seleccion.csv`.

Resultado: hay siete huecos, concentrados en dos ventanas. La primera va de
finales de abril a mediados de mayo y afecta a Pancostales, La Culebra y Los
Ceres. La segunda va de principios a finales de junio y afecta a las cuatro
parcelas. Puri no tiene hueco en la primera ventana porque entre sus
observaciones hay 13 días. El mayor hueco de cada parcela es de 23, 20, 23 y 18
días, igual que el obtenido en el notebook 03.

In [ ]:
huecos = []
for parcela in PARCELAS_ESPERADAS:
    fechas = series[parcela].loc[series[parcela]["seleccionada"], "date"].reset_index(drop=True)
    dias = fechas.diff().dt.days
    for i in dias[dias > UMBRAL_CORTE_DIAS].index:
        huecos.append({"parcela": parcela,
                       "desde": fechas[i - 1].strftime("%Y-%m-%d"),
                       "hasta": fechas[i].strftime("%Y-%m-%d"),
                       "dias": int(dias[i])})

huecos_seleccion = pd.DataFrame(huecos)
display(huecos_seleccion)
huecos_seleccion.to_csv(CARPETA_GRAFICAS / "huecos_seleccion.csv", index=False, encoding="utf-8-sig")

for ruta in rutas_graficas:
    print("Guardada:", ruta.name, "|", round(ruta.stat().st_size / 1024), "KB")

## 3.5 Tabla de valores de los índices

Para describir las figuras con datos, se calcula para cada parcela e índice el
valor inicial, el máximo con su fecha, el mínimo con su fecha y el valor final
de las observaciones seleccionadas. La tabla se guarda en
`resumen_indices_seleccionadas.csv`.

Resultado:
- El máximo de NDVI, NDMI y EVI se alcanza el 04/07/2026 en Pancostales, Puri
  y La Culebra y el 29/06/2026 en Los Ceres. El NDRE alcanza su máximo el
  29/06/2026 en las cuatro parcelas.
- El NDVI máximo va de 0,815 (Pancostales) a 0,938 (La Culebra); el EVI, de
  0,750 a 0,974; el NDMI, de 0,384 a 0,486.
- El NDVI mínimo (de 0,095 a 0,188) se da entre mediados de abril y finales de
  mayo. El NDMI parte de valores negativos y termina otra vez en negativos.
- El NDRE máximo va de 0,580 (Puri) a 0,731 (La Culebra) y su mínimo, de 0,048
  a 0,128, se da entre mediados de abril y finales de mayo. Parte de valores
  entre 0,111 y 0,180 y termina entre 0,133 y 0,196.
- Las cuatro parcelas siguen un patrón parecido: valores bajos en abril y mayo,
  subida en junio, máximo a finales de junio o principios de julio, descenso en
  julio y agosto, y valores bajos y estables en septiembre.

Estos resultados describen el comportamiento de los índices. No se infiere el
estado hídrico ni el cultivo, que no está confirmado.

In [ ]:
filas = []
for parcela in PARCELAS_ESPERADAS:
    sel = series[parcela][series[parcela]["seleccionada"]].reset_index(drop=True)
    for indice in INDICES_GRAFICAS:
        col = f"{indice}_mean"
        i_max, i_min = sel[col].idxmax(), sel[col].idxmin()
        filas.append({
            "parcela": parcela,
            "indice": indice,
            "valor_inicial": round(sel[col].iloc[0], 3),
            "maximo": round(sel.loc[i_max, col], 3),
            "fecha_maximo": sel.loc[i_max, "date"].strftime("%Y-%m-%d"),
            "minimo": round(sel.loc[i_min, col], 3),
            "fecha_minimo": sel.loc[i_min, "date"].strftime("%Y-%m-%d"),
            "valor_final": round(sel[col].iloc[-1], 3),
        })

resumen_indices = pd.DataFrame(filas)
display(resumen_indices)
resumen_indices.to_csv(CARPETA_GRAFICAS / "resumen_indices_seleccionadas.csv", index=False, encoding="utf-8-sig")

## 3.6 Comprobación del caso de Los Ceres

En la figura del NDVI, Los Ceres presenta tras el máximo un mínimo local hacia
finales de julio y una recuperación en agosto, a diferencia de las otras tres
parcelas. Antes de describirlo se comprueba la cobertura de parcela de las
observaciones de ese tramo.

Resultado: la observación del 24/07/2026 (NDVI 0,584) tiene una cobertura del
85,0 %, por encima del umbral del 70 % usado en la selección. Dos días después,
el 26/07, el NDVI es 0,697 con una cobertura del 96,1 %, y el 03/08 llega a
0,781. Las observaciones con cobertura baja (19/07, 29/07 y 08/08) no están
seleccionadas. Por tanto, el mínimo local no se explica por una cobertura por
debajo del umbral. La causa queda sin determinar y no se interpreta.

`NDVI_count` y `pixeles_antes` no son directamente comparables, como advierte
el notebook 03.

In [ ]:
tabla_ceres = series["los_ceres"]
col_cob = columna_que_contiene(tabla_ceres, "coberturaminima")
revision_ceres = tabla_ceres.loc[
    (tabla_ceres["date"] >= "2026-07-10") & (tabla_ceres["date"] <= "2026-08-10"),
    ["date", "seleccionada", col_cob, "NDVI_mean", "NDVI_count"],
]
display(revision_ceres)

## 3.7 Conclusiones de la Parte 3

- Se han representado NDVI, NDRE, NDMI y EVI de las cuatro parcelas para el periodo
  provisional, con un panel por parcela, el mismo eje vertical, títulos,
  etiquetas y leyendas.
- Las figuras distinguen las observaciones seleccionadas de las no seleccionadas
  y cortan la línea en los huecos de más de 14 días. No se interpola.
- Los cuatro índices muestran en las cuatro parcelas un máximo a finales de
  junio o principios de julio, precedido de valores bajos y seguido de un
  descenso. El NDRE lo alcanza el 29/06 en las cuatro parcelas; los demás, el
  04/07 en tres de ellas.
- Los huecos de observación se concentran en dos ventanas, de finales de abril a
  mediados de mayo y de junio, y esta segunda coincide con la subida de los
  índices.
- Los Ceres presenta un mínimo local a finales de julio que no se explica por la
  cobertura; queda sin determinar.
- NDRE se incorporó al extractor del notebook 03 en la issue #4 y se representa
  igual que el resto de índices.
- Los índices describen la vegetación. No miden la humedad del suelo y el cultivo
  no está confirmado, por lo que no se emiten interpretaciones agronómicas.